# Phase 3: ML Modeling - Dataset Preparation
## DNA Gene Mapping Project
**Author:** Sharique Mohammad  
**Date:** February 2026  

---

## Objective
Prepare ML-ready datasets with:
- **Chunked stratified sampling** (memory-efficient for 8GB RAM)
- **Class imbalance handling** (SMOTE)
- **Feature scaling** (StandardScaler)
- **Train/validation/test splits** verification

## Tasks
1. **Variant Pathogenicity Prediction** (Binary Classification)
2. **Structural Variant Risk Prediction** (Binary Classification)

---
## 1. Setup and Configuration

In [ ]:
# Imports
import pandas as pd
import numpy as np
import psycopg2
from pathlib import Path
import json
import time
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import pickle
import warnings
warnings.filterwarnings('ignore')

print(" Imports successful")

In [ ]:
# Configuration
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "ml"
FEATURE_DIR = PROJECT_ROOT / "data" / "analytical" / "feature_lists"
MODEL_DIR = PROJECT_ROOT / "models"

# Create directories
for dir_path in [DATA_DIR, MODEL_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Sampling configuration
TRAIN_SAMPLE_FRAC = 0.3  # 30% of training data (~870K rows)
CHUNK_SIZE = 500000      # Load 500K rows per chunk
RANDOM_STATE = 42
SMOTE_STRATEGY = 0.5     # Oversample minority to 50% of majority

print("="*80)
print("PHASE 3: ML DATASET PREPARATION")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Train sample fraction: {TRAIN_SAMPLE_FRAC*100:.0f}%")
print(f"Chunk size: {CHUNK_SIZE:,} rows")
print(f"Random state: {RANDOM_STATE}")
print("="*80)

---
## 2. Database Connection

In [ ]:
# PostgreSQL connection
from dotenv import load_dotenv
import os
load_dotenv()

POSTGRES_HOST = os.getenv('POSTGRES_HOST', 'localhost')
POSTGRES_PORT = os.getenv('POSTGRES_PORT', '5432')
POSTGRES_DB = os.getenv('POSTGRES_DB', 'genome_db')
POSTGRES_USER = os.getenv('POSTGRES_USER', 'postgres')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')

print("Connecting to PostgreSQL...")
try:
    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        database=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD
    )
    print(" Connected successfully")
except Exception as e:
    print(f" Connection failed: {e}")
    raise

---
## 3. Load Feature Lists from Phase 2

In [ ]:
def load_feature_list(table_name):
    """Load final feature list for a table"""
    file_path = FEATURE_DIR / f"{table_name}_final_features.csv"
    if not file_path.exists():
        print(f" Feature list not found: {file_path}")
        return []
    features = pd.read_csv(file_path)['feature_name'].tolist()
    return features

# Load feature lists
clinical_features = load_feature_list('clinical_ml_features')
disease_features = load_feature_list('disease_ml_features')
pharmacogene_features = load_feature_list('pharmacogene_ml_features')
impact_features = load_feature_list('variant_impact_ml_features')
sv_features = load_feature_list('structural_variant_ml_features')

print("Feature counts from Phase 2:")
print(f"  Clinical: {len(clinical_features)} features")
print(f"  Disease: {len(disease_features)} features")
print(f"  Pharmacogene: {len(pharmacogene_features)} features")
print(f"  Impact: {len(impact_features)} features")
print(f"  SV: {len(sv_features)} features")

---
## 4. Chunked Stratified Sampling Functions

In [ ]:
def load_stratified_sample(conn, table, target_col, sample_frac=0.3, chunk_size=500000):
    """
    Load stratified sample in chunks to avoid memory issues.
    
    Args:
        conn: PostgreSQL connection
        table: Table name
        target_col: Target column name for stratification
        sample_frac: Fraction to sample (0.3 = 30%)
        chunk_size: Rows per chunk
    
    Returns:
        DataFrame with stratified sample
    """
    start_time = time.time()
    
    # Step 1: Get total count
    total = pd.read_sql(f"SELECT COUNT(*) FROM gold.{table}", conn).iloc[0, 0]
    print(f"\n{table}:")
    print(f"  Total rows: {total:,}")
    
    # Step 2: Load IDs and target only (small!)
    print(f"  Loading metadata for stratification...", end=" ", flush=True)
    id_col = 'variant_id' if 'variant' in table else 'sv_id'
    meta = pd.read_sql(f"""
        SELECT 
            {id_col} as id,
            {target_col}
        FROM gold.{table}
        WHERE {target_col} IS NOT NULL
    """, conn)
    print(f"{len(meta):,} rows")
    
    # Check class distribution
    class_dist = meta[target_col].value_counts()
    print(f"  Class distribution:")
    for cls, count in class_dist.items():
        print(f"    {cls}: {count:,} ({count/len(meta)*100:.1f}%)")
    
    # Step 3: Stratified sample
    print(f"  Stratified sampling ({sample_frac*100:.0f}%)...", end=" ", flush=True)
    sample_ids, _ = train_test_split(
        meta['id'],
        stratify=meta[target_col],
        train_size=sample_frac,
        random_state=RANDOM_STATE
    )
    sample_ids_set = set(sample_ids)
    print(f"{len(sample_ids):,} samples selected")
    
    # Step 4: Load in chunks
    print(f"  Loading features in chunks:")
    chunks = []
    n_chunks = (total // chunk_size) + 1
    
    for i, offset in enumerate(range(0, total, chunk_size), 1):
        chunk = pd.read_sql(f"""
            SELECT * FROM gold.{table}
            ORDER BY {id_col}
            LIMIT {chunk_size} OFFSET {offset}
        """, conn)
        
        # Keep only sampled IDs
        sampled = chunk[chunk[id_col].isin(sample_ids_set)]
        chunks.append(sampled)
        
        mem_usage = sampled.memory_usage(deep=True).sum() / 1e6
        print(f"    Chunk {i}/{n_chunks}: kept {len(sampled):,} rows ({mem_usage:.1f} MB)")
        
        del chunk
    
    # Step 5: Merge
    df = pd.concat(chunks, ignore_index=True)
    
    elapsed = time.time() - start_time
    mem_usage = df.memory_usage(deep=True).sum() / 1e9
    print(f"  Final: {len(df):,} rows, {mem_usage:.2f} GB, {elapsed:.1f}s")
    
    return df

In [ ]:
def load_full_dataset(conn, table):
    """Load full validation/test set in chunks"""
    start_time = time.time()
    
    # Get total count
    total = pd.read_sql(f"SELECT COUNT(*) FROM gold.{table}", conn).iloc[0, 0]
    print(f"\n{table}:")
    print(f"  Total rows: {total:,}")
    print(f"  Loading in chunks:")
    
    chunks = []
    n_chunks = (total // CHUNK_SIZE) + 1
    id_col = 'variant_id' if 'variant' in table else 'sv_id'
    
    for i, offset in enumerate(range(0, total, CHUNK_SIZE), 1):
        chunk = pd.read_sql(f"""
            SELECT * FROM gold.{table}
            ORDER BY {id_col}
            LIMIT {CHUNK_SIZE} OFFSET {offset}
        """, conn)
        chunks.append(chunk)
        
        mem_usage = chunk.memory_usage(deep=True).sum() / 1e6
        print(f"    Chunk {i}/{n_chunks}: {len(chunk):,} rows ({mem_usage:.1f} MB)")
    
    df = pd.concat(chunks, ignore_index=True)
    
    elapsed = time.time() - start_time
    mem_usage = df.memory_usage(deep=True).sum() / 1e9
    print(f"   Loaded: {len(df):,} rows, {mem_usage:.2f} GB, {elapsed:.1f}s")
    
    return df

---
## 5. Task 1: Variant Pathogenicity - Load Data

In [ ]:
print("="*80)
print("TASK 1: VARIANT PATHOGENICITY PREDICTION")
print("="*80)

# Load training set (stratified sample)
train_variants = load_stratified_sample(
    conn, 
    'ml_dataset_variants_train',
    'target_is_pathogenic',
    sample_frac=TRAIN_SAMPLE_FRAC
)

In [ ]:
# Load validation set (full)
val_variants = load_full_dataset(conn, 'ml_dataset_variants_validation')

In [ ]:
# Load test set (full)
test_variants = load_full_dataset(conn, 'ml_dataset_variants_test')

In [ ]:
# Verify class distributions
print("\nClass distribution verification:")
for name, df in [('Train', train_variants), ('Val', val_variants), ('Test', test_variants)]:
    dist = df['target_is_pathogenic'].value_counts()
    total = len(df[df['target_is_pathogenic'].notna()])
    print(f"  {name}: ", end="")
    for cls in sorted(dist.index):
        count = dist[cls]
        pct = count / total * 100
        print(f"{cls}={count:,} ({pct:.1f}%), ", end="")
    print()

---
## 6. Feature Preparation

In [ ]:
# Combine all variant feature lists (remove duplicates)
all_features = list(set(
    clinical_features + disease_features + 
    pharmacogene_features + impact_features
))
print(f"Total unique features: {len(all_features)}")

# Target column
target_col = 'target_is_pathogenic'

In [ ]:
def prepare_features(df, features, target):
    """Extract features and target, handle missing values"""
    from sklearn.preprocessing import LabelEncoder
    
    # Get available features
    available = [f for f in features if f in df.columns]
    missing = set(features) - set(available)
    
    if missing:
        print(f"  Warning: {len(missing)} features not found in data")
    
    X = df[available].copy()
    y = df[target].copy()
    
    # Remove rows with missing target
    valid_idx = y.notna()
    X = X[valid_idx]
    y = y[valid_idx]
    
    # Convert boolean columns to numeric (0/1)
    bool_cols = X.select_dtypes(include=['bool']).columns
    for col in bool_cols:
        X[col] = X[col].astype('int8')
    
    # Define categorical features to encode (from Phase 2 feature lists)
    categorical_features = [
        'protein_impact_category', 'inheritance_pattern', 'x_linked_risk_modifier',
        'inheritance_pathogenicity_modifier', 'gene_mutation_burden', 'gene_variant_profile',
        'variant_type', 'domain_impact_severity', 'conservation_impact_class',
        'pharmacogene_category', 'gene_pharmacogene_priority', 'disease_complexity',
        'sv_type_class', 'sv_size_category', 'gene_impact_severity'  # For SVs
    ]
    
    # Encode categorical features
    for col in categorical_features:
        if col in X.columns:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str).fillna('Missing'))
    
    # Drop ID/metadata columns
    id_cols = ['variant_id', 'sv_id', 'gene_name', 'chromosome', 'position', 
               'official_gene_symbol', 'clinical_significance_simple', 
               'clinvar_pathogenicity_class', 'disease_enriched', 'primary_disease',
               'omim_id', 'mondo_id', 'orphanet_id', 'disease_name_enriched']
    
    for col in id_cols:
        if col in X.columns:
            X = X.drop(columns=[col])
    
    # Drop any remaining object columns (unrecognized text)
    object_cols = X.select_dtypes(include=['object']).columns.tolist()
    if object_cols:
        print(f"  Dropping {len(object_cols)} unrecognized text columns: {object_cols[:5]}")
        X = X.drop(columns=object_cols)
    
    # Handle missing values in numeric features
    for col in X.columns:
        if X[col].isna().any():
            X[col].fillna(X[col].median(), inplace=True)
    
    return X, y, X.columns.tolist()

In [ ]:
# Save feature list
feature_list_file = DATA_DIR / "variant_features_used.txt"
with open(feature_list_file, 'w') as f:
    for feat in sorted(train_features):
        f.write(f"{feat}\n")
print(f" Feature list saved: {feature_list_file.name}")

---
## 7. Feature Scaling

In [ ]:
print("\nFeature scaling (StandardScaler)...")

# Fit on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print(f"  Scaled all feature sets")

# Save scaler
scaler_file = MODEL_DIR / "variant_scaler.pkl"
with open(scaler_file, 'wb') as f:
    pickle.dump(scaler, f)
print(f" Scaler saved: {scaler_file.name}")

---
## 8. Class Imbalance Handling (SMOTE)

In [ ]:
print("="*80)
print("CLASS IMBALANCE HANDLING")
print("="*80)

# Check imbalance
class_counts = y_train.value_counts()
imbalance_ratio = class_counts[False] / class_counts[True]
print(f"Original class distribution:")
print(f"  Class 0 (Benign): {class_counts[False]:,}")
print(f"  Class 1 (Pathogenic): {class_counts[True]:,}")
print(f"  Imbalance ratio: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 2:
    print(f"\nApplying SMOTE (strategy={SMOTE_STRATEGY})...")
    smote = SMOTE(sampling_strategy=SMOTE_STRATEGY, random_state=RANDOM_STATE)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
    
    new_counts = pd.Series(y_train_balanced).value_counts()
    print(f"  After SMOTE:")
    print(f"    Class 0: {new_counts[False]:,}")
    print(f"    Class 1: {new_counts[True]:,}")
    print(f"    New ratio: {new_counts[False]/new_counts[True]:.2f}:1")
    
    # Save both versions
    datasets = {
        'original': (X_train_scaled, y_train),
        'balanced': (X_train_balanced, y_train_balanced)
    }
else:
    print("  Classes reasonably balanced, SMOTE not needed")
    datasets = {
        'original': (X_train_scaled, y_train)
    }

---
## 9. Save Variant Datasets

In [ ]:
print("\n" + "="*80)
print("SAVING VARIANT DATASETS")
print("="*80)

# Save training sets
for version, (X, y) in datasets.items():
    train_file = DATA_DIR / f"variant_train_{version}.pkl"
    with open(train_file, 'wb') as f:
        pickle.dump({'X': X, 'y': y}, f)
    print(f"  Saved: {train_file.name} ({len(y):,} samples)")

# Save validation set
val_file = DATA_DIR / "variant_validation.pkl"
with open(val_file, 'wb') as f:
    pickle.dump({'X': X_val_scaled, 'y': y_val}, f)
print(f" Saved: {val_file.name} ({len(y_val):,} samples)")

# Save test set
test_file = DATA_DIR / "variant_test.pkl"
with open(test_file, 'wb') as f:
    pickle.dump({'X': X_test_scaled, 'y': y_test}, f)
print(f"   Saved: {test_file.name} ({len(y_test):,} samples)")

---
## 10. Task 2: Structural Variant Risk - Load Data

In [ ]:
print("\n" + "="*80)
print("TASK 2: STRUCTURAL VARIANT RISK PREDICTION")
print("="*80)

# Load SV datasets (full data - small enough)
print("\nLoading SV datasets (full data - small enough)...")
sv_train = pd.read_sql("SELECT * FROM gold.ml_dataset_structural_variants_train", conn)
sv_val = pd.read_sql("SELECT * FROM gold.ml_dataset_structural_variants_validation", conn)
sv_test = pd.read_sql("SELECT * FROM gold.ml_dataset_structural_variants_test", conn)

print(f"  Train: {len(sv_train):,} rows")
print(f"  Val: {len(sv_val):,} rows")
print(f"  Test: {len(sv_test):,} rows")

In [ ]:
# Prepare SV features
sv_target_col = 'is_high_risk_sv'

X_sv_train, y_sv_train, sv_train_features = prepare_features(sv_train, sv_features, sv_target_col)
X_sv_val, y_sv_val, _ = prepare_features(sv_val, sv_features, sv_target_col)
X_sv_test, y_sv_test, _ = prepare_features(sv_test, sv_features, sv_target_col)

print(f"\nSV feature matrices:")
print(f"  Train: X={X_sv_train.shape}, y={len(y_sv_train)}")
print(f"  Val: X={X_sv_val.shape}, y={len(y_sv_val)}")
print(f"  Test: X={X_sv_test.shape}, y={len(y_sv_test)}")

# Check class balance
sv_class_counts = y_sv_train.value_counts()
print(f"\nSV class distribution:")
print(f"  Low-risk (0): {sv_class_counts.get(False, 0):,}")
print(f"  High-risk (1): {sv_class_counts.get(True, 0):,}")
if len(sv_class_counts) > 1:
    ratio = max(sv_class_counts) / min(sv_class_counts)
    print(f"  Imbalance ratio: {ratio:.2f}:1 (reasonable)")

In [ ]:
# Scale SV features
print("\nScaling SV features...")
sv_scaler = StandardScaler()
X_sv_train_scaled = sv_scaler.fit_transform(X_sv_train)
X_sv_val_scaled = sv_scaler.transform(X_sv_val)
X_sv_test_scaled = sv_scaler.transform(X_sv_test)

X_sv_train_scaled = pd.DataFrame(X_sv_train_scaled, columns=X_sv_train.columns)
X_sv_val_scaled = pd.DataFrame(X_sv_val_scaled, columns=X_sv_val.columns)
X_sv_test_scaled = pd.DataFrame(X_sv_test_scaled, columns=X_sv_test.columns)

print("  Scaled all SV feature sets")

---
## 11. Save SV Datasets

In [ ]:
# Save SV datasets
sv_train_file = DATA_DIR / "sv_train.pkl"
with open(sv_train_file, 'wb') as f:
    pickle.dump({'X': X_sv_train_scaled, 'y': y_sv_train}, f)

sv_val_file = DATA_DIR / "sv_validation.pkl"
with open(sv_val_file, 'wb') as f:
    pickle.dump({'X': X_sv_val_scaled, 'y': y_sv_val}, f)

sv_test_file = DATA_DIR / "sv_test.pkl"
with open(sv_test_file, 'wb') as f:
    pickle.dump({'X': X_sv_test_scaled, 'y': y_sv_test}, f)

sv_scaler_file = MODEL_DIR / "sv_scaler.pkl"
with open(sv_scaler_file, 'wb') as f:
    pickle.dump(sv_scaler, f)

print(f" Saved all SV datasets:")
print(f"  - {sv_train_file.name}")
print(f"  - {sv_val_file.name}")
print(f"  - {sv_test_file.name}")
print(f"  - {sv_scaler_file.name}")

---
## 12. Summary Report

In [ ]:
print("\n" + "="*80)
print("SUMMARY REPORT")
print("="*80)

summary = {
    "timestamp": datetime.now().isoformat(),
    "configuration": {
        "train_sample_fraction": TRAIN_SAMPLE_FRAC,
        "chunk_size": CHUNK_SIZE,
        "random_state": RANDOM_STATE,
        "smote_strategy": SMOTE_STRATEGY
    },
    "variant_pathogenicity": {
        "training": {
            "samples": len(y_train),
            "features": len(train_features),
            "class_0": int(class_counts[False]),
            "class_1": int(class_counts[True]),
            "imbalance_ratio": float(imbalance_ratio)
        },
        "validation": {
            "samples": len(y_val),
            "features": X_val.shape[1]
        },
        "test": {
            "samples": len(y_test),
            "features": X_test.shape[1]
        }
    },
    "sv_risk": {
        "training": {
            "samples": len(y_sv_train),
            "features": len(sv_train_features)
        },
        "validation": {
            "samples": len(y_sv_val)
        },
        "test": {
            "samples": len(y_sv_test)
        }
    }
}

# Add balanced training info if exists
if 'y_train_balanced' in locals():
    summary['variant_pathogenicity']['balanced_training'] = {
        "samples": len(y_train_balanced),
        "class_0": int(new_counts[False]),
        "class_1": int(new_counts[True])
    }

# Save summary
summary_file = DATA_DIR / "dataset_preparation_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print("\nVARIANT PATHOGENICITY DATASETS:")
print(f"  Training (original): {len(y_train):,} samples, {len(train_features)} features")
if 'y_train_balanced' in locals():
    print(f"  Training (SMOTE): {len(y_train_balanced):,} samples")
print(f"  Validation: {len(y_val):,} samples")
print(f"  Test: {len(y_test):,} samples")

print("\nSTRUCTURAL VARIANT DATASETS:")
print(f"  Training: {len(y_sv_train):,} samples, {len(sv_train_features)} features")
print(f"  Validation: {len(y_sv_val):,} samples")
print(f"  Test: {len(y_sv_test):,} samples")

print(f"\n Summary saved: {summary_file.name}")

In [ ]:
# List all created files
print("\nFILES CREATED:")
print(f"  {DATA_DIR.relative_to(PROJECT_ROOT)}")
for file in sorted(DATA_DIR.glob("*.pkl")):
    size_mb = file.stat().st_size / 1e6
    print(f"    - {file.name} ({size_mb:.1f} MB)")
print(f"  {MODEL_DIR.relative_to(PROJECT_ROOT)}")
for file in sorted(MODEL_DIR.glob("*_scaler.pkl")):
    print(f"    - {file.name}")

---
## 13. Cleanup

In [ ]:
conn.close()
print(" Database connection closed")